# Phase 3 — Data Cleaning & Transformation

This notebook begins Phase 3 with a pre-cleaning investigation of the raw hospital encounter data. No cleaning, transformation, or export is performed at this stage.

## 1. Imports and Raw Data Loading

In [2]:
import pandas as pd
from pathlib import Path

# Support running the notebook from either the project root or notebooks folder.
project_root = Path.cwd()
if not (project_root / "data" / "raw" / "diabetic_data.csv").exists():
    project_root = project_root.parent

raw_data_path = project_root / "data" / "raw" / "diabetic_data.csv"

# Load the raw file without changing its values. Literal question marks remain unchanged.
df_raw = pd.read_csv(raw_data_path)

print(f"Loaded raw data from: {raw_data_path}")
print(f"Dataset shape: {df_raw.shape}")

Loaded raw data from: /Users/josmithareddy/Documents/Data-Analyst-Projects/hospital-readmission-risk-analysis/data/raw/diabetic_data.csv
Dataset shape: (101766, 50)


## 2. Raw Data Integrity Check

These checks verify that this notebook starts from the same raw dataset examined during Phase 2.

In [3]:
total_rows, total_columns = df_raw.shape
unique_encounter_ids = df_raw["encounter_id"].nunique(dropna=False)
unique_patients = df_raw["patient_nbr"].nunique(dropna=False)
fully_duplicated_rows = df_raw.duplicated().sum()

print(f"Total rows: {total_rows:,}")
print(f"Total columns: {total_columns:,}")
print(f"Unique encounter IDs: {unique_encounter_ids:,}")
print(f"Unique patients: {unique_patients:,}")
print(f"Fully duplicated rows: {fully_duplicated_rows:,}")

Total rows: 101,766
Total columns: 50
Unique encounter IDs: 101,766
Unique patients: 71,518
Fully duplicated rows: 0


## 3. Missing Value Audit

The audit counts standard null/NaN values and literal `?` placeholders separately, then combines them for review. It does not replace either representation.

In [4]:
standard_null_counts = df_raw.isna().sum()
question_mark_counts = df_raw.eq("?").sum()
combined_missing_counts = standard_null_counts + question_mark_counts

missing_audit = pd.DataFrame({
    "column": df_raw.columns,
    "standard_null_count": standard_null_counts.values,
    "question_mark_count": question_mark_counts.values,
    "combined_missing_count": combined_missing_counts.values,
    "combined_missing_percentage": (
        combined_missing_counts.values / len(df_raw) * 100
    ),
})

# Keep only columns that contain at least one of the two missing-value forms.
missing_audit = (
    missing_audit.loc[missing_audit["combined_missing_count"] > 0]
    .sort_values("combined_missing_percentage", ascending=False)
    .reset_index(drop=True)
)

print("Missing-value audit:")
print(missing_audit.to_string(index=False, formatters={
    "combined_missing_percentage": lambda value: f"{value:.2f}%"
}))

Missing-value audit:
           column  standard_null_count  question_mark_count  combined_missing_count combined_missing_percentage
           weight                    0                98569                   98569                      96.86%
    max_glu_serum                96420                    0                   96420                      94.75%
        A1Cresult                84748                    0                   84748                      83.28%
medical_specialty                    0                49949                   49949                      49.08%
       payer_code                    0                40256                   40256                      39.56%
             race                    0                 2273                    2273                       2.23%
           diag_3                    0                 1423                    1423                       1.40%
           diag_2                    0                  358                     358

## 4. Investigate Question-Mark Column Discrepancy

Phase 2 contained a discrepancy: one detailed output appeared to show 6 columns containing `?`, while a later summary reported 7. The following fresh calculation reports the raw-data result directly without hiding or automatically reconciling that discrepancy.

In [5]:
question_mark_columns = question_mark_counts[question_mark_counts > 0].index.tolist()

question_mark_audit = pd.DataFrame({
    "column": question_mark_columns,
    "question_mark_count": question_mark_counts.loc[question_mark_columns].values,
    "question_mark_percentage": (
        question_mark_counts.loc[question_mark_columns].values / len(df_raw) * 100
    ),
}).sort_values("question_mark_percentage", ascending=False)

print(f"Columns containing at least one literal '?': {question_mark_columns}")
print(f"Total columns containing literal '?': {len(question_mark_columns)}")
print("Question-mark counts and percentages for every affected column:")
print(question_mark_audit.to_string(index=False, formatters={
    "question_mark_percentage": lambda value: f"{value:.2f}%"
}))

Columns containing at least one literal '?': ['race', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3']
Total columns containing literal '?': 7
Question-mark counts and percentages for every affected column:
           column  question_mark_count question_mark_percentage
           weight                98569                   96.86%
medical_specialty                49949                   49.08%
       payer_code                40256                   39.56%
             race                 2273                    2.23%
           diag_3                 1423                    1.40%
           diag_2                  358                    0.35%
           diag_1                   21                    0.02%


## 5. High-Missingness Column Value Investigation

For this investigation only, temporary masks treat both standard null/NaN values and literal `?` placeholders as missing. The values in `df_raw` remain unchanged.

In [6]:
columns_to_investigate = [
    "weight",
    "medical_specialty",
    "payer_code",
    "race",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult",
]

for column in columns_to_investigate:
    series = df_raw[column]
    missing_mask = series.isna() | series.eq("?")
    observed_values = series.loc[~missing_mask]
    missing_percentage = missing_mask.mean() * 100
    unique_non_missing_values = observed_values.nunique()
    top_10_values = observed_values.value_counts().head(10)

    print(f"\nColumn: {column}")
    print(f"Missing percentage: {missing_percentage:.2f}%")
    print(f"Unique non-missing values: {unique_non_missing_values:,}")
    print("Top 10 observed non-missing values and counts:")
    print(top_10_values.to_string())


Column: weight
Missing percentage: 96.86%
Unique non-missing values: 9
Top 10 observed non-missing values and counts:
weight
[75-100)     1336
[50-75)       897
[100-125)     625
[125-150)     145
[25-50)        97
[0-25)         48
[150-175)      35
[175-200)      11
>200            3

Column: medical_specialty
Missing percentage: 49.08%
Unique non-missing values: 72
Top 10 observed non-missing values and counts:
medical_specialty
InternalMedicine              14635
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Radiologist                    1140
Pulmonology                     871

Column: payer_code
Missing percentage: 39.56%
Unique non-missing values: 17
Top 10 observed non-missing values and counts:
payer_code
MC    32439
HM     6274
SP     5007
BC     4655
MD     3532
CP     2

## 6. Cleaning Decision Table

This table documents the approved cleaning decisions. These decisions are recorded here only; no cleaning actions are executed in this section.

In [ ]:
# Document approved decisions only. This cell does not alter df_raw.
cleaning_decisions = pd.DataFrame([
    {
        "column": "weight",
        "issue": "96.86% missing",
        "missing_percentage": 96.86,
        "proposed_action": "Drop column",
        "reason": "Only approximately 3.14% of encounter records contain a recorded weight, making coverage too limited for the planned analysis.",
        "status": "Approved",
    },
    {
        "column": "medical_specialty",
        "issue": "49.08% missing represented by '?'",
        "missing_percentage": 49.08,
        "proposed_action": "Keep and convert missing placeholder to 'Unknown'",
        "reason": "Approximately half of the records contain specialty information and the variable may be useful for analyzing readmission patterns by provider specialty.",
        "status": "Approved",
    },
    {
        "column": "payer_code",
        "issue": "39.56% missing represented by '?'",
        "missing_percentage": 39.56,
        "proposed_action": "Drop column",
        "reason": "High missingness and limited relevance to the primary clinical readmission questions in this project.",
        "status": "Approved",
    },
    {
        "column": "race",
        "issue": "2.23% missing represented by '?'",
        "missing_percentage": 2.23,
        "proposed_action": "Keep and convert missing placeholder to 'Unknown'",
        "reason": "Most records contain race information and the field is useful for demographic analysis.",
        "status": "Approved",
    },
    {
        "column": "diag_1",
        "issue": "0.02% missing represented by '?'",
        "missing_percentage": 0.02,
        "proposed_action": "Keep and convert missing placeholder to 'Unknown'",
        "reason": "Primary diagnosis is analytically important and missingness is negligible.",
        "status": "Approved",
    },
    {
        "column": "diag_2",
        "issue": "0.35% missing represented by '?'",
        "missing_percentage": 0.35,
        "proposed_action": "Keep and convert missing placeholder to 'Unknown'",
        "reason": "Secondary diagnosis has very high coverage and may provide useful clinical information.",
        "status": "Approved",
    },
    {
        "column": "diag_3",
        "issue": "1.40% missing represented by '?'",
        "missing_percentage": 1.40,
        "proposed_action": "Keep and convert missing placeholder to 'Unknown'",
        "reason": "Additional diagnosis information has high coverage and may contribute to clinical analysis.",
        "status": "Approved",
    },
    {
        "column": "max_glu_serum",
        "issue": "94.75% standard missing values",
        "missing_percentage": 94.75,
        "proposed_action": "Keep as categorical and represent missing results as 'Not Recorded'",
        "reason": "Although sparsely populated, recorded glucose test results may contain useful information. Missing status must not be interpreted as a normal result.",
        "status": "Approved",
    },
    {
        "column": "A1Cresult",
        "issue": "83.28% standard missing values",
        "missing_percentage": 83.28,
        "proposed_action": "Keep as categorical and represent missing results as 'Not Recorded'",
        "reason": "A1C is relevant to diabetes-related analysis. Missing status must remain distinguishable from an actual normal test result.",
        "status": "Approved",
    },
])

print("Cleaning decision table:")
print(cleaning_decisions.to_string(index=False, formatters={
    "missing_percentage": lambda value: f"{value:.2f}%"
}))

## 7. Categorical Data Audit

This section inspects important categorical variables before any cleaning or transformation is performed. All calculations read from `df_raw`; no values are replaced, removed, or changed.

### A. Inspect Core Categorical Variables

In [ ]:
core_categorical_columns = ["gender", "age", "change", "diabetesMed", "readmitted"]

for column in core_categorical_columns:
    # Include every observed value, including standard missing values if present.
    counts = df_raw[column].value_counts(dropna=False)
    percentages = counts / len(df_raw) * 100
    value_summary = pd.DataFrame({
        "count": counts,
        "percentage_of_total": percentages,
    })

    print(f"\nColumn: {column}")
    print(f"Unique values: {df_raw[column].unique().tolist()}")
    print(value_summary.to_string(formatters={
        "percentage_of_total": lambda value: f"{value:.2f}%"
    }))

### B. Inspect Coded Hospital Variables

`admission_type_id`, `discharge_disposition_id`, and `admission_source_id` are categorical codes even though Pandas may store them as integers. This audit reports the observed codes without interpreting or mapping their numeric values.

In [ ]:
coded_hospital_columns = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]

for column in coded_hospital_columns:
    counts = df_raw[column].value_counts(dropna=False).sort_index()
    percentages = counts / len(df_raw) * 100
    code_summary = pd.DataFrame({
        "count": counts,
        "percentage_of_total": percentages,
    })

    print(f"\nColumn: {column}")
    print(f"Sorted unique values: {sorted(df_raw[column].dropna().unique().tolist())}")
    print(code_summary.to_string(formatters={
        "percentage_of_total": lambda value: f"{value:.2f}%"
    }))

### C. Inspect Diagnosis Fields

Diagnosis codes are inspected in their raw form. No grouping, conversion, or other transformation is performed.

In [ ]:
diagnosis_columns = ["diag_1", "diag_2", "diag_3"]

for column in diagnosis_columns:
    question_mark_count = df_raw[column].eq("?").sum()
    non_missing_mask = df_raw[column].notna() & df_raw[column].ne("?")
    top_20_codes = df_raw.loc[non_missing_mask, column].value_counts().head(20)

    print(f"\nColumn: {column}")
    print(f"Number of unique values: {df_raw[column].nunique(dropna=False):,}")
    print(f"Number of literal '?' values: {question_mark_count:,}")
    print("Top 20 most common non-missing codes:")
    print(top_20_codes.to_string())

### D. Inspect Diabetes-Related Categorical Variables

The following value counts include standard missing values. Missing values are reported as observed and are not replaced.

In [ ]:
diabetes_categorical_columns = [
    "insulin",
    "change",
    "diabetesMed",
    "max_glu_serum",
    "A1Cresult",
]

for column in diabetes_categorical_columns:
    print(f"\nColumn: {column}")
    print(df_raw[column].value_counts(dropna=False).to_string())

### E. Audit Summary

The summary below flags observed representations for further review. A label is not classified as erroneous merely because it appears unusual. Unexpected-value checks are limited to the low-cardinality fields for which explicit reference sets are defined in the code.

In [ ]:
# Combine the audited column lists while preserving their original order.
categorical_audit_columns = list(dict.fromkeys(
    core_categorical_columns
    + coded_hospital_columns
    + diagnosis_columns
    + diabetes_categorical_columns
))

columns_with_unknown_labels = [
    column for column in categorical_audit_columns
    if df_raw[column].isin(["Unknown", "Unknown/Invalid"]).any()
]
columns_with_question_marks = [
    column for column in categorical_audit_columns
    if df_raw[column].eq("?").any()
]
columns_with_standard_nan = [
    column for column in categorical_audit_columns
    if df_raw[column].isna().any()
]

# Reference sets support reporting only; they do not change or remove values.
expected_values = {
    "gender": {"Female", "Male", "Unknown/Invalid"},
    "age": {
        "[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
        "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)",
    },
    "change": {"Ch", "No"},
    "diabetesMed": {"Yes", "No"},
    "readmitted": {"NO", ">30", "<30"},
    "insulin": {"No", "Steady", "Up", "Down"},
    "max_glu_serum": {"Norm", ">200", ">300"},
    "A1Cresult": {"Norm", ">7", ">8"},
}

unexpected_values = {}
for column, expected in expected_values.items():
    observed = set(df_raw[column].dropna().unique())
    unexpected = sorted(observed - expected)
    if unexpected:
        unexpected_values[column] = unexpected

print("Categorical audit summary:")
print(f"Columns containing 'Unknown' or 'Unknown/Invalid': {columns_with_unknown_labels}")
print(f"Columns containing literal '?': {columns_with_question_marks}")
print(f"Columns containing standard NaN values: {columns_with_standard_nan}")
print(f"Values outside the defined reference sets (review only): {unexpected_values}")
print("Values reported for review are not automatically classified as erroneous.")

## 7. Pre-Cleaning Investigation Summary

The raw dataset has not been modified. Missingness can be represented as either standard null/NaN values or literal `?` placeholders. High missingness alone is not sufficient reason to automatically drop a variable. Cleaning decisions will be made only after reviewing each field's analytical relevance and data quality.

The next step is to finalize the cleaning decision table.